# Assesing distribution of roads

Este notebook evalúa la distribución de vialidades y la ubicación de las cámaras de velocidad en diferentes configuraciones de cuadrícula (tamaños y offsets). El objetivo es seleccionar la mejor configuración de cuadrícula que minimice la variabilidad en la distribución de vialidades y la distancia de las cámaras al centroide de cada celda.

**Inputs necesarios:**
- `vialidades.json`: Geometría de las vialidades de la Ciudad de México
- `fotocivicas-ubicacion-puntos/fotocivicas-ubicacion-puntos.shp`: Ubicación de las cámaras de velocidad
- `central-tendencies-grid-sizes.parquet`: Tendencias centrales calculadas previamente para diferentes tamaños de grid

**Outputs generados:**
- Análisis exploratorio de diferentes configuraciones de grid (sin archivos guardados explícitamente)

In [2]:
import os
import sys
sys.path.insert(1, '../')

import math
import warnings
import numpy as np
import pandas as pd
import seaborn as sns
import geopandas as gpd
import matplotlib.pyplot as plt
from utils import build_grid
from shapely.geometry import LineString, MultiLineString, Polygon


warnings.filterwarnings("ignore")
PATH_DATA = '../../data/'
CRS = 6372

In [3]:
vialidades = (
    gpd
    .read_file(os.path.join(PATH_DATA, 'vialidades.json'))
    .assign(
        tipo_vialidad=lambda x: x.TIPO_VIA.map({
            "Vía primaria":"primaria", 
            'Vía de acceso controlado':'acceso_controlado'
        }),
        carriles=lambda x: pd.to_numeric(x.CARRILES, errors='raise'),
        sentidos=lambda x: x.CIRCULA.map({
            "Un sentido":1, 
            "Dos sentidos":2, 
            "Un sentido con carril de contraflujo":2
        })
    )
    .rename({
        'NIVEL':'nivel', 
        'NOMENCLAT':'calle', 
        'NOMBRE':'vialidad', 
        'ID_VIA':'id_via'
    }, axis=1)
    [[
        'id_via', 'vialidad', 'calle', 'tipo_vialidad', 
        'carriles', 'sentidos', 'nivel', 'geometry'
    ]]
    .to_crs(CRS)
)

speed_cameras = gpd.read_file(
    os.path.join(
        PATH_DATA,
        'fotocivicas-ubicacion-puntos',
        'fotocivicas-ubicacion-puntos.shp'
    )
).to_crs(CRS)

# Ya tenemos todos los datos que necesitamos
tendencies = pd.read_parquet(os.path.join(PATH_DATA, "central-tendencies-grid-sizes.parquet"))

In [4]:
def calcular_road_in_grid(grid, grid_size, offset_x, offset_y):
    result = (
        # Intersección de las vialidades con la cuadrícula
        gpd
        .overlay(vialidades, grid, how='intersection')
        .assign(length_m=lambda x: x.length)
        .groupby("grid_id")
        .agg(total_road_in_grid=pd.NamedAgg('length_m', 'sum'))
        .assign(
            identifier=f"{grid_size}:{offset_x}:{offset_y}"
        )
        .reset_index(drop=True)
    )
    return result

In [5]:
def calcular_distancia_camaras_centroide(grid, grid_size, offset_x, offset_y, grid_length):
    # La distancia máxima es aquella en la que la cámara está en una de las esquinas del cuadrado
    # De la esquina del cuadrado al centro hay la siguiente distancia
    distancia_maxima = math.sqrt(2*grid_length*grid_length)/2
    result = (
        gpd
        .sjoin(speed_cameras, grid, how='inner')
        .drop(columns=['centroid', 'index_right'])
        .merge(
            grid.set_geometry('centroid').to_crs(CRS).drop(columns='geometry'),
            on='grid_id',
            how='inner'
        )
        .rename({
            'geometry':'loc_camera',
            'centroid':'center_grid'
        }, axis=1)
        .assign(
            # es distancia relativa (para ser comparables)
            distance_to_center=lambda x: x.loc_camera.distance(x.center_grid)/distancia_maxima,
            identifier=f"{grid_size}:{offset_x}:{offset_y}"
        )
        [["identifier", "distance_to_center"]]
    )
    return result

In [6]:
offset_every = 10
grid_sizes = [123, 185, 368, 500]
tendencias = []

for grid_size in grid_sizes:
    break
    grid_length = build_grid(vialidades, grid_size).get('length_m')
    max_offset = int(grid_length//2)

    offsets = []
    for x in range(0, max_offset, offset_every):
        for y in range(0, max_offset, offset_every):
            offsets.append({"x":x, "y":y})
            
    for offset in offsets:
        print(grid_size, offset, end='\r')
        grid = build_grid(
            df=vialidades,
            n=grid_size,
            offset_x_m=offset.get('x'),
            offset_y_m=offset.get('y')
        ).get('grid')

        grid_tendencia_central = (
            calcular_road_in_grid(
                grid,
                grid_size,
                offset.get('x'),
                offset.get('y')
            )
            .groupby('identifier')
            .total_road_in_grid
            .agg(["mean", "std"])
            .reset_index()
            .assign(grid_size=grid_size)
        )

        distance_tendencias = (
            calcular_distancia_camaras_centroide(
                grid, 
                grid_size, 
                offset.get('x'), 
                offset.get('y'), 
                grid_length
            )
            .groupby('identifier')
            .agg(
                median_dtc=pd.NamedAgg('distance_to_center', 'median'),
                mean_dtc=pd.NamedAgg('distance_to_center', 'mean'),
                std_dtc=pd.NamedAgg('distance_to_center', 'std'),
            )
            .reset_index()
        )
        tendencias.append(grid_tendencia_central.merge(distance_tendencias, on='identifier'))

In [7]:
# Ahora lo que tenemos que hacer es elegir el mejor offset para cada uno de los grid sizes
color_map = {
    123:'#264653',
    185:'#2a9d8f',
    368:'#fca311',
    500:'#e76f51'
}

def clean_ax(ax):
    ax.spines.top.set_visible(False)
    ax.spines.right.set_visible(False)

    axis_color = "#6c757d"
    ax.xaxis.label.set_color(axis_color)
    ax.yaxis.label.set_color(axis_color)
    ax.spines.bottom.set_color(axis_color)
    ax.spines.left.set_color(axis_color)
    ax.tick_params(axis='both', colors=axis_color)
    ax.grid(axis='y', alpha=.2, linestyle=':')
    
    ttl = ax.title
    ttl.set_position([.5, 2])

fig, axes = plt.subplots(2, 2, figsize=(10,10))
axes = axes.reshape(-1)
fig.suptitle("Media y STD en celdas con desfaces", ha="left", color="gray", x=0.07, y=.975)


for ax, grid_size in zip(axes, grid_sizes):
    clean_ax(ax)

    dfgs = tendencies.query('grid_size == @grid_size')
    sizes = dfgs.median_dtc.max() + dfgs.median_dtc.min() - dfgs.median_dtc
    sizes = sizes - sizes.mean()
    sizes[sizes < 0] = max(sizes[sizes > 0].min(), 0.01)
    sizes = np.exp(sizes*130)
    sizes[sizes > 100] = 100

    (
        dfgs
        .plot.scatter(
            ax=ax,
            x='mean',
            y='std',
            s=sizes,
            color=color_map.get(grid_size),
            alpha=1,
            linewidth=0
        )
    )
    ax.set_ylabel("STD kms in grid")
    ax.set_xlabel("Mean kms in grid")
    ax.set_title(f"Grid Size {grid_size}", color='darkgray', loc='left', fontsize=10)

fig.tight_layout()
# fig.savefig(os.path.join(PATH_DATA, "graphs", "mean-std-on-offset-grids.png"), dpi=300, transparent=True)
plt.close()

In [29]:
# Y ahora ya nada más es tema de elegir cuáles son los offsets que se van a usar para el análisis
# Para cada grid size vamos a elgir tres opciones: la que minimice el median, el mean y el std
np.random.seed(188192)
measures = ['median_dtc', 'mean_dtc', 'std_dtc']
response = {}

grid_size = grid_sizes[0]
for grid_size in grid_sizes:
    response[grid_size] = {}

    dfgs = tendencies.query('grid_size == @grid_size')
    for measure in measures:
        _, offset_x, offset_y = dfgs.sort_values(measure, ignore_index=True).iloc[0].identifier.split(':')
        response[grid_size][measure] = {"x":int(offset_x), "y":int(offset_y)}
        
    # Dejamos el caso base
    response[grid_size]['base'] = {"x":0, "y":0}
    
    # Agregamos un caso aleatorio
    _, offset_x, offset_y = np.random.choice(dfgs.identifier).split(':')
    response[grid_size]['random'] = {"x":int(offset_x), "y":int(offset_y)}

In [32]:
from pprint import pprint
pprint(response)

{123: {'base': {'x': 0, 'y': 0},
       'mean_dtc': {'x': 130, 'y': 80},
       'median_dtc': {'x': 140, 'y': 110},
       'random': {'x': 30, 'y': 80},
       'std_dtc': {'x': 0, 'y': 80}},
 185: {'base': {'x': 0, 'y': 0},
       'mean_dtc': {'x': 0, 'y': 60},
       'median_dtc': {'x': 0, 'y': 50},
       'random': {'x': 0, 'y': 10},
       'std_dtc': {'x': 80, 'y': 20}},
 368: {'base': {'x': 0, 'y': 0},
       'mean_dtc': {'x': 30, 'y': 0},
       'median_dtc': {'x': 20, 'y': 0},
       'random': {'x': 40, 'y': 20},
       'std_dtc': {'x': 40, 'y': 40}},
 500: {'base': {'x': 0, 'y': 0},
       'mean_dtc': {'x': 20, 'y': 0},
       'median_dtc': {'x': 20, 'y': 0},
       'random': {'x': 10, 'y': 20},
       'std_dtc': {'x': 30, 'y': 20}}}


In [31]:
pd.DataFrame(response)

,123,185,368,500
median_dtc,"{'x': 140, 'y': 110}","{'x': 0, 'y': 50}","{'x': 20, 'y': 0}","{'x': 20, 'y': 0}"
mean_dtc,"{'x': 130, 'y': 80}","{'x': 0, 'y': 60}","{'x': 30, 'y': 0}","{'x': 20, 'y': 0}"
std_dtc,"{'x': 0, 'y': 80}","{'x': 80, 'y': 20}","{'x': 40, 'y': 40}","{'x': 30, 'y': 20}"
base,"{'x': 0, 'y': 0}","{'x': 0, 'y': 0}","{'x': 0, 'y': 0}","{'x': 0, 'y': 0}"
random,"{'x': 30, 'y': 80}","{'x': 0, 'y': 10}","{'x': 40, 'y': 20}","{'x': 10, 'y': 20}"
